
#  InsightForge: AI-Powered Business Intelligence System

##  Introduction
This project demonstrates how Artificial Intelligence can be integrated with Business Intelligence to generate actionable insights from both structured and unstructured data.

The system combines:
-  Structured data (CSV sales dataset)
-  Unstructured data (business report PDF)
-  Large Language Models (LLMs)
-  Retrieval-Augmented Generation (RAG)

Key capabilities include:
- Natural language querying of business data
- Context-aware responses using memory
- Multi-source intelligence (CSV + PDF)
- AI-generated insights and recommendations

This solution enables organizations to make faster, smarter, and data-driven decisions.

1️⃣ Imports & Setup

In [1]:
# ==============================
# 1️⃣ Imports & Setup
# ==============================
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
import warnings
warnings.filterwarnings("ignore")  # hide warnings for clean output
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import ConversationalRetrievalChain

##  Data Loading & Exploration

In [2]:
# Load CSV
df = pd.read_csv("sales_data.csv")
display(df.head(10))

,Date,Product,Region,Sales,Customer_Age,Customer_Gender,Customer_Satisfaction
0,2022-01-01,Widget C,South,786,26,Male,2.874407
1,2022-01-02,Widget D,East,850,29,Male,3.365205
2,2022-01-03,Widget A,North,871,40,Female,4.547364
3,2022-01-04,Widget C,South,464,31,Male,4.555420
4,2022-01-05,Widget C,South,262,50,Female,3.982935
5,2022-01-06,Widget D,East,147,35,Female,1.140628
6,2022-01-07,Widget A,North,610,66,Male,3.300282
7,2022-01-08,Widget A,South,428,58,Male,4.146334
8,2022-01-09,Widget C,West,939,26,Male,1.069152
9,2022-01-10,Widget B,West,215,40,Male,3.198738


## Data Visualization

In [3]:
# Unique filter options
regions = ["All"] + list(df["Region"].unique())
products = ["All"] + list(df["Product"].unique())

# Dropdown widgets
region_dropdown = widgets.Dropdown(options=regions, value="All", description="Region:")
product_dropdown = widgets.Dropdown(options=products, value="All", description="Product:")

display(region_dropdown, product_dropdown)

Dropdown(description='Region:', options=('All', 'South', 'East', 'North', 'West'), value='All')

Dropdown(description='Product:', options=('All', 'Widget C', 'Widget D', 'Widget A', 'Widget B'), value='All')

In [4]:
# Function to update dashboard based on filters
def update_dashboard(region, product):
    clear_output(wait=True)  # clear previous output
    display(region_dropdown, product_dropdown)  # keep dropdowns visible
    
    # Filter data
    filtered_df = df.copy()
    if region != "All":
        filtered_df = filtered_df[filtered_df["Region"] == region]
    if product != "All":
        filtered_df = filtered_df[filtered_df["Product"] == product]
    
    # ==============================
    # KPIs
    # ==============================
    total_sales = filtered_df["Sales"].sum()
    top_product = filtered_df.groupby("Product")["Sales"].sum().idxmax() if not filtered_df.empty else "N/A"
    top_region = filtered_df.groupby("Region")["Sales"].sum().idxmax() if not filtered_df.empty else "N/A"
    avg_satisfaction = filtered_df["Customer_Satisfaction"].mean().round(2) if not filtered_df.empty else 0
    avg_customer_age = filtered_df["Customer_Age"].mean().round(1) if not filtered_df.empty else 0
    
    display(Markdown("## 📊 Key Metrics"))
    display(Markdown(f"- **Total Sales:** ${total_sales:,.0f}"))
    display(Markdown(f"- **Top Product:** {top_product}"))
    display(Markdown(f"- **Top Region:** {top_region}"))
    display(Markdown(f"- **Average Customer Satisfaction:** {avg_satisfaction} / 5"))
    display(Markdown(f"- **Average Customer Age:** {avg_customer_age}"))
    
    # ==============================
    # Charts
    # ==============================
    # Sales by Product
    display(Markdown("### Sales by Product"))
    sales_by_product = filtered_df.groupby("Product")["Sales"].sum()
    fig, ax = plt.subplots()
    sales_by_product.plot(kind='bar', ax=ax, color='skyblue')
    ax.set_ylabel("Sales")
    ax.set_title("Total Sales by Product")
    plt.show()
    
    # Sales by Region
    display(Markdown("### Sales by Region"))
    sales_by_region = filtered_df.groupby("Region")["Sales"].sum()
    fig, ax = plt.subplots()
    sales_by_region.plot(kind='bar', ax=ax, color='lightgreen')
    ax.set_ylabel("Sales")
    ax.set_title("Total Sales by Region")
    plt.show()
    
    # Customer Satisfaction by Product
    display(Markdown("### Customer Satisfaction by Product"))
    satisfaction_by_product = filtered_df.groupby("Product")["Customer_Satisfaction"].mean()
    fig, ax = plt.subplots()
    satisfaction_by_product.plot(kind='bar', ax=ax, color='orange')
    ax.set_ylabel("Avg Satisfaction")
    ax.set_title("Customer Satisfaction by Product")
    plt.show()

These visualizations help identify top-performing products and regions.

In [5]:
# Connect dropdowns to update function
widgets.interactive(update_dashboard, region=region_dropdown, product=product_dropdown)

interactive(children=(Dropdown(description='Region:', options=('All', 'South', 'East', 'North', 'West'), value…

 AI-Powered Business Intelligence Q&A System

In [6]:
# Convert dataset rows to text for AI retriever
data_texts = df.apply(
    lambda row: f"Date: {row['Date']}, Product: {row['Product']}, Region: {row['Region']}, Sales: {row['Sales']}, Customer Age: {row['Customer_Age']}, Customer Gender: {row['Customer_Gender']}, Customer Satisfaction: {row['Customer_Satisfaction']}",
    axis=1
).tolist()

# Create vector store for RAG
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(data_texts, embeddings)

# Initialize AI LLM
llm = ChatOpenAI(temperature=0.7, model="gpt-3.5-turbo")
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vectorstore.as_retriever())

In [7]:
pdf_loader = PyPDFLoader("business_report.pdf")
pdf_docs = pdf_loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

pdf_texts = text_splitter.split_documents(pdf_docs)
pdf_text_strings = [
    f"This is from a business report document: {doc.page_content}"
    for doc in pdf_texts
]
all_texts = data_texts + pdf_text_strings
vectorstore = FAISS.from_texts(all_texts, embeddings)
pdf_vectorstore = FAISS.from_texts(pdf_text_strings, embeddings)
pdf_retriever = pdf_vectorstore.as_retriever(search_kwargs={"k": 3})

pdf_qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=pdf_retriever,
    memory=memory
)

## Memory Test (Multi-turn Conversation)

In [8]:
question = "Which product has the highest sales?"
result = qa({"query": question})
print(result["result"])

question = "Why is it performing well?"
result = qa({"query": question})
print(result["result"])

Widget A has the highest sales with 988 units sold in the South region on 2024-05-11.
Based on the provided data, Widget A seems to be performing well in terms of sales. The customer satisfaction scores for Widget A are relatively high (1.04 and 1.32), which could indicate that customers are happy with the product. However, without more specific information or data, it is difficult to pinpoint the exact reasons why Widget A is performing well.


In [9]:
question = "What recommendations are mentioned in the business report?"
result = pdf_qa({"question": question})
print(result["answer"])

The recommendations mentioned in the business report are:
1. Increase marketing for Widget A
2. Improve customer experience for low-performing regions


We now build an AI system that can answer questions based on the dataset using embeddings and retrieval.

## 💬 Interactive AI Assistant

In [ ]:
display(Markdown("## 💬 Ask AI about your Business Data"))

while True:
    question = input("Type your question (or 'exit' to quit): ")
    
    if question.lower() == "exit":
        print("Exiting AI assistant. ✅")
        break
    
    result = qa({"query": question})
    print("AI Answer:", result["result"])

## 💬 Ask AI about your Business Data

Filter-aware AI Q&A

1️⃣ Function to create filtered retriever

In [ ]:
def get_filtered_qa(filtered_df):
    # Convert filtered rows to text
    data_texts = filtered_df.apply(
        lambda row: f"Date: {row['Date']}, Product: {row['Product']}, Region: {row['Region']}, Sales: {row['Sales']}, Customer Age: {row['Customer_Age']}, Customer Gender: {row['Customer_Gender']}, Customer Satisfaction: {row['Customer_Satisfaction']}",
        axis=1
    ).tolist()
    
    # Create vectorstore retriever
    vectorstore = FAISS.from_texts(all_texts, embeddings)
from langchain.chains import ConversationalRetrievalChain

qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    memory=memory
)

2️⃣ Function to run interactive AI with dropdown filters

In [ ]:
def interactive_ai(region, product):
    # Filter data based on current selections
    filtered_df = df.copy()
    if region != "All":
        filtered_df = filtered_df[filtered_df["Region"] == region]
    if product != "All":
        filtered_df = filtered_df[filtered_df["Product"] == product]
    
    # Create filter-aware QA
    qa_filtered = get_filtered_qa(filtered_df)
    
    display(Markdown(f"### 💬 Ask AI about {region} / {product} data"))
    
    while True:
        question = input("Type your question (or 'exit' to quit): ")
        if question.lower() == "exit":
            print("Exiting AI assistant. ✅")
            break
        answer = qa_filtered.run(question)
        display(Markdown(f"**AI Answer:** {answer}"))

3️⃣ Connect AI to dashboard dropdowns

In [ ]:
# Example: run AI for current selections
widgets.interactive(interactive_ai, region=region_dropdown, product=product_dropdown)

In [ ]:
from langchain.evaluation.qa import QAEvalChain

We evaluate the AI's responses against expected answers to measure accuracy and reliability.

## AI Evaluation (QAEvalChain)

This section evaluates the accuracy of the AI-generated responses using predefined ground truth answers.

In [ ]:
eval_data = [
    {
        "query": "Which product has the highest sales?",
        "answer": "Widget A has the highest total sales."
    },
    {
        "query": "Which region performs the best?",
        "answer": "The region with the highest total sales performs the best."
    }
]

In [ ]:
predictions = []

for item in eval_data:
    result = qa({"query": item["query"]})
    predictions.append({"result": result["result"]})

In [ ]:
eval_chain = QAEvalChain.from_llm(llm)

graded_outputs = eval_chain.evaluate(
    eval_data,
    predictions
)

In [ ]:
for i, output in enumerate(graded_outputs):
    print(f"Question: {eval_data[i]['query']}")
    print(f"Expected: {eval_data[i]['answer']}")
    print(f"AI Answer: {predictions[i]['result']}")
    print(f"Evaluation: {output['text']}")
    print("-" * 50)

##  Conclusion

This project successfully demonstrates the implementation of an AI-powered Business Intelligence system using modern LLM and retrieval techniques.

Key achievements:
- Built an interactive AI assistant for business queries
- Enabled contextual understanding using conversation memory
- Integrated structured (CSV) and unstructured (PDF) data sources
- Implemented Retrieval-Augmented Generation (RAG)
- Designed a scalable architecture for real-world BI applications

The system highlights how AI can transform traditional dashboards into intelligent decision-making tools.

Future enhancements may include:
- Real-time data integration
- Advanced visual dashboards
- Deployment as a web application

 This project showcases the practical application of AI in solving real business problems.